# ColabProSST

ColabProSST adds the ProSST-2048 backbone to the SaprotHub/OPMC workflow. It uses amino-acid `input_ids` plus ProSST `ss_input_ids`; it does not use SaProt AA+3Di merged tokens. The notebook supports structure-token conversion, zero-shot mutation scoring, classification/regression fine-tuning, and checkpoint-based prediction.


In [ ]:
#@title 1. Install ColabProSST
import os
import subprocess
import sys
from pathlib import Path

SAPROTHUB_REPO = 'https://github.com/westlake-repl/SaprotHub' #@param {type:'string'}
SAPROTHUB_BRANCH = 'prosst' #@param {type:'string'}

ROOT = Path('/content')
SAPROT_HOME = ROOT / 'SaprotHub'
PROSST_HOME = ROOT / 'ProSST'
UPLOAD_HOME = ROOT / 'prosst_uploads'
ASSET_HOME = ROOT / 'prosst_structure_assets'
PYG_HOME = ROOT / '.cache' / 'pyg'
HF_HOME = ROOT / '.cache' / 'huggingface'
UPLOAD_HOME.mkdir(parents=True, exist_ok=True)
ASSET_HOME.mkdir(parents=True, exist_ok=True)
PYG_HOME.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['PYG_HOME'] = str(PYG_HOME)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if not SAPROT_HOME.exists():
    try:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', SAPROTHUB_BRANCH, SAPROTHUB_REPO, str(SAPROT_HOME)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'Could not clone {SAPROTHUB_REPO}@{SAPROTHUB_BRANCH}. '
            'Push the ColabProSST branch or set SAPROTHUB_REPO/SAPROTHUB_BRANCH '
            'to a repository that contains this notebook and the ProSST modules.'
        ) from exc
if not PROSST_HOME.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ai4protein/ProSST', str(PROSST_HOME)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SAPROT_HOME / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'joblib==1.3.2', 'pathos==0.3.2', 'biotite==0.39.0'], check=True)
import torch
pyg_backend = 'cu121' if torch.version.cuda else 'cpu'
pyg_wheel_url = f'https://data.pyg.org/whl/torch-2.2.0+{pyg_backend}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv', '-f', pyg_wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric==2.5.0'], check=True)

os.environ['PROSST_HOME'] = str(PROSST_HOME)
for path in [str(SAPROT_HOME), str(SAPROT_HOME / 'saprot'), str(PROSST_HOME)]:
    if path not in sys.path:
        sys.path.insert(0, path)

def maybe_upload_path(current_path, upload_enabled, upload_dir=UPLOAD_HOME):
    current_path = str(current_path).strip()
    if current_path:
        return current_path
    if not upload_enabled:
        raise ValueError('Set a file path or enable upload for this cell.')
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab file upload is only available in Google Colab.') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    saved_paths = []
    for filename, content in uploaded.items():
        safe_name = Path(filename).name
        save_path = upload_dir / safe_name
        save_path.write_bytes(content)
        saved_paths.append(save_path)
    return str(saved_paths[0])



def maybe_extract_asset_zip(zip_path='', upload_enabled=False, extract_dir=ASSET_HOME):
    zip_path = str(zip_path).strip()
    if not zip_path and not upload_enabled:
        return None
    if not zip_path:
        zip_path = maybe_upload_path('', True, upload_dir=UPLOAD_HOME)
    import zipfile
    archive_path = Path(zip_path)
    if not archive_path.exists():
        raise FileNotFoundError(f'Structure asset zip does not exist: {archive_path}')
    import shutil
    target_dir = extract_dir / archive_path.stem
    shutil.rmtree(target_dir, ignore_errors=True)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_root = target_dir.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_target = (target_dir / member.filename).resolve()
            if target_root not in [member_target, *member_target.parents]:
                raise ValueError(f'Unsafe zip member path: {member.filename}')
            archive.extract(member, target_dir)
    print('extracted structure assets to', target_dir)
    return str(target_dir)


def attach_last_structure_tokens(csv_path, output_path):
    if 'LAST_PROSST_STRUCTURE_TOKENS' not in globals():
        raise RuntimeError('Run the structure token conversion cell first, or provide structure_tokens/structure_path/pdb_path in the CSV.')
    import pandas as pd
    df = pd.read_csv(csv_path)
    lower_columns = {column.lower(): column for column in df.columns}
    has_structure = any(column in lower_columns for column in ['structure_tokens', 'structure_path', 'pdb_path'])
    if not has_structure:
        sequence_column = lower_columns.get('sequence', lower_columns.get('protein'))
        if sequence_column is None:
            raise ValueError('CSV needs a sequence/protein column before reusing the last structure tokens.')
        sequences = df[sequence_column].astype(str).str.strip().str.upper()
        expected_sequence = str(LAST_PROSST_SEQUENCE).strip().upper()
        mismatched = sequences[sequences != expected_sequence]
        if not mismatched.empty:
            raise ValueError(
                'Cannot reuse one structure token sequence for CSV rows with different protein sequences. '
                'Add per-row structure_tokens/structure_path/pdb_path columns, or run with a single matching sequence.'
            )
        df['structure_tokens'] = ' '.join(str(token) for token in LAST_PROSST_STRUCTURE_TOKENS)
    df.to_csv(output_path, index=False)
    return output_path

print('ColabProSST is ready.')

In [ ]:
#@title 2. Create CSV templates
DOWNLOAD_TEMPLATES = False #@param {type:'boolean'}

import zipfile
from pathlib import Path
import pandas as pd

TEMPLATE_HOME = Path('/content/prosst_templates')
TEMPLATE_HOME.mkdir(parents=True, exist_ok=True)
for old_template in TEMPLATE_HOME.glob('prosst_*_template.csv'):
    old_template.unlink()

pd.DataFrame([
    {'sequence': 'ACD', 'mutant': 'D3A', 'structure_tokens': '0 1 2'},
    {'sequence': 'ACDE', 'mutant': 'D3A:E4A', 'structure_tokens': '0 1 2 3'},
]).to_csv(TEMPLATE_HOME / 'prosst_zero_shot_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'mutant': 'D3A', 'pdb_path': 'protein.pdb', 'chain_id': ''},
]).to_csv(TEMPLATE_HOME / 'prosst_zero_shot_pdb_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'label': 1, 'stage': 'train', 'structure_tokens': '0 1 2'},
    {'sequence': 'ACE', 'label': 0, 'stage': 'valid', 'structure_tokens': '0 1 3'},
    {'sequence': 'ACF', 'label': 1, 'stage': 'test', 'structure_tokens': '0 1 4'},
]).to_csv(TEMPLATE_HOME / 'prosst_classification_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'label': 1, 'stage': 'train', 'pdb_path': 'train.pdb', 'chain_id': ''},
    {'sequence': 'ACE', 'label': 0, 'stage': 'valid', 'pdb_path': 'valid.pdb', 'chain_id': ''},
    {'sequence': 'ACF', 'label': 1, 'stage': 'test', 'pdb_path': 'test.pdb', 'chain_id': ''},
]).to_csv(TEMPLATE_HOME / 'prosst_classification_pdb_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'label': 0.5, 'stage': 'train', 'structure_tokens': '0 1 2'},
    {'sequence': 'ACE', 'label': 0.2, 'stage': 'valid', 'structure_tokens': '0 1 3'},
    {'sequence': 'ACF', 'label': 0.8, 'stage': 'test', 'structure_tokens': '0 1 4'},
]).to_csv(TEMPLATE_HOME / 'prosst_regression_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'label': 0.5, 'stage': 'train', 'pdb_path': 'train.pdb', 'chain_id': ''},
    {'sequence': 'ACE', 'label': 0.2, 'stage': 'valid', 'pdb_path': 'valid.pdb', 'chain_id': ''},
    {'sequence': 'ACF', 'label': 0.8, 'stage': 'test', 'pdb_path': 'test.pdb', 'chain_id': ''},
]).to_csv(TEMPLATE_HOME / 'prosst_regression_pdb_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'structure_tokens': '0 1 2'},
    {'sequence': 'ACE', 'structure_tokens': '0 1 3'},
]).to_csv(TEMPLATE_HOME / 'prosst_prediction_template.csv', index=False)

pd.DataFrame([
    {'sequence': 'ACD', 'pdb_path': 'protein_1.pdb', 'chain_id': ''},
    {'sequence': 'ACE', 'pdb_path': 'protein_2.pdb', 'chain_id': ''},
]).to_csv(TEMPLATE_HOME / 'prosst_prediction_pdb_template.csv', index=False)

template_zip = TEMPLATE_HOME / 'prosst_csv_templates.zip'
with zipfile.ZipFile(template_zip, 'w') as archive:
    for csv_path in sorted(TEMPLATE_HOME.glob('*.csv')):
        archive.write(csv_path, arcname=csv_path.name)

print('template directory:', TEMPLATE_HOME)
print('template zip:', template_zip)
if DOWNLOAD_TEMPLATES:
    try:
        from google.colab import files
        files.download(str(template_zip))
    except Exception:
        pass


In [ ]:
#@title 3. Convert PDB/CIF to ProSST structure tokens
STRUCTURE_PATH = '' #@param {type:'string'}
UPLOAD_STRUCTURE = False #@param {type:'boolean'}
CHAIN_ID = '' #@param {type:'string'}
CACHE_DIR = '/content/prosst_structure_cache' #@param {type:'string'}
STRUCTURE_TOKEN_CSV = '/content/prosst_structure_tokens.csv' #@param {type:'string'}
DOWNLOAD_STRUCTURE_TOKEN_CSV = False #@param {type:'boolean'}

from saprot.data.pdb2prosst import load_or_quantize_structure, serialize_structure_tokens

STRUCTURE_PATH = maybe_upload_path(STRUCTURE_PATH, UPLOAD_STRUCTURE)
chain = CHAIN_ID.strip() or None
result = load_or_quantize_structure(STRUCTURE_PATH, cache_dir=CACHE_DIR, chain_id=chain)
import pandas as pd
LAST_PROSST_STRUCTURE = result
LAST_PROSST_SEQUENCE = result['sequence']
LAST_PROSST_STRUCTURE_TOKENS = result['structure_tokens']
LAST_PROSST_STRUCTURE_PATH = STRUCTURE_PATH
LAST_PROSST_CHAIN_ID = chain or ''
pd.DataFrame([{
    'sequence': LAST_PROSST_SEQUENCE,
    'structure_tokens': serialize_structure_tokens(LAST_PROSST_STRUCTURE_TOKENS),
    'pdb_path': LAST_PROSST_STRUCTURE_PATH,
    'chain_id': LAST_PROSST_CHAIN_ID,
}]).to_csv(STRUCTURE_TOKEN_CSV, index=False)
print('sequence length:', len(result['sequence']))
print('structure token length:', len(result['structure_tokens']))
print('first 20 tokens:', result['structure_tokens'][:20])
print('saved structure token csv:', STRUCTURE_TOKEN_CSV)
if DOWNLOAD_STRUCTURE_TOKEN_CSV:
    try:
        from google.colab import files
        files.download(STRUCTURE_TOKEN_CSV)
    except Exception:
        pass

In [ ]:
#@title 4. Zero-shot mutation effect prediction
MUTATION_CSV = '' #@param {type:'string'}
UPLOAD_MUTATION_CSV = False #@param {type:'boolean'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
OUTPUT_CSV = '/content/prosst_mutation_scores.csv' #@param {type:'string'}
DOWNLOAD_MUTATION_RESULTS = True #@param {type:'boolean'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}

from saprot.scripts.mutation_zeroshot_prosst import score_mutants

MUTATION_CSV = maybe_upload_path(MUTATION_CSV, UPLOAD_MUTATION_CSV)
structure_base_dir = maybe_extract_asset_zip(STRUCTURE_ZIP, UPLOAD_STRUCTURE_ZIP)
if USE_LAST_STRUCTURE_TOKENS:
    MUTATION_CSV = attach_last_structure_tokens(MUTATION_CSV, '/content/prosst_mutation_with_structure.csv')
df = score_mutants(
    input_csv=MUTATION_CSV,
    output_csv=OUTPUT_CSV,
    model_path=MODEL_PATH,
    cache_dir='/content/prosst_structure_cache',
    structure_base_dir=structure_base_dir,
)
display(df.head())
print('saved to', OUTPUT_CSV)
if DOWNLOAD_MUTATION_RESULTS:
    try:
        from google.colab import files
        files.download(OUTPUT_CSV)
    except Exception:
        pass

In [ ]:
#@title 5. Fine-tune classification/regression
TASK_TYPE = 'classification' #@param ['classification', 'regression']
TRAIN_CSV = '' #@param {type:'string'}
UPLOAD_TRAIN_CSV = False #@param {type:'boolean'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
TASK_NAME = 'ProSSTUserTask' #@param {type:'string'}
NUM_LABELS = 2 #@param {type:'integer'}
MAX_EPOCHS = 2 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
TEST_RESULT_CSV = '/content/prosst_test_predictions.csv' #@param {type:'string'}
FREEZE_BACKBONE = False #@param {type:'boolean'}
GRADIENT_CHECKPOINTING = True #@param {type:'boolean'}
DOWNLOAD_MODEL_CHECKPOINT = False #@param {type:'boolean'}

import torch
from pathlib import Path
from easydict import EasyDict
from saprot.utils.construct_prosst_lmdb import construct_prosst_lmdb
from saprot.utils.module_loader import my_load_dataset, my_load_model, load_trainer

TRAIN_CSV = maybe_upload_path(TRAIN_CSV, UPLOAD_TRAIN_CSV)
structure_base_dir = maybe_extract_asset_zip(STRUCTURE_ZIP, UPLOAD_STRUCTURE_ZIP)
if USE_LAST_STRUCTURE_TOKENS:
    TRAIN_CSV = attach_last_structure_tokens(TRAIN_CSV, '/content/prosst_train_with_structure.csv')
LMDB_HOME = Path('/content/SaprotHub/LMDB')
WEIGHT_HOME = Path('/content/SaprotHub/weights/prosst')
WEIGHT_HOME.mkdir(parents=True, exist_ok=True)

construct_prosst_lmdb(
    TRAIN_CSV,
    str(LMDB_HOME),
    TASK_NAME,
    TASK_TYPE,
    cache_dir='/content/prosst_structure_cache',
    structure_base_dir=structure_base_dir,
)

model_py = 'prosst/prosst_classification_model' if TASK_TYPE == 'classification' else 'prosst/prosst_regression_model'
dataset_py = 'prosst/prosst_classification_dataset' if TASK_TYPE == 'classification' else 'prosst/prosst_regression_dataset'
checkpoint_path = WEIGHT_HOME / f'{TASK_NAME}.pt'
model_kwargs = {
    'config_path': MODEL_PATH,
    'load_pretrained': True,
    'freeze_backbone': FREEZE_BACKBONE,
    'gradient_checkpointing': GRADIENT_CHECKPOINTING,
    'save_path': str(checkpoint_path),
    'test_result_path': TEST_RESULT_CSV,
    'lr_scheduler_kwargs': {'class': 'ConstantLRScheduler', 'init_lr': 2.0e-5},
    'optimizer_kwargs': {'class': 'AdamW', 'betas': [0.9, 0.98], 'weight_decay': 0.01},
}
if TASK_TYPE == 'classification':
    model_kwargs['num_labels'] = NUM_LABELS

config = EasyDict({
    'model': {'model_py_path': model_py, 'kwargs': model_kwargs},
    'dataset': {
        'dataset_py_path': dataset_py,
        'train_lmdb': str(LMDB_HOME / TASK_NAME / 'train'),
        'valid_lmdb': str(LMDB_HOME / TASK_NAME / 'valid'),
        'test_lmdb': str(LMDB_HOME / TASK_NAME / 'test'),
        'dataloader_kwargs': {'batch_size': BATCH_SIZE, 'num_workers': 0},
        'kwargs': {'tokenizer': MODEL_PATH},
    },
    'Trainer': {
        'max_epochs': MAX_EPOCHS,
        'log_every_n_steps': 1,
        'strategy': {'class': 'auto'},
        'logger': False,
        'enable_checkpointing': False,
        'val_check_interval': 1.0,
        'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
        'devices': 1,
        'num_nodes': 1,
        'accumulate_grad_batches': 1,
        'precision': 16 if torch.cuda.is_available() else 32,
        'num_sanity_val_steps': 0,
    },
})

model = my_load_model(config.model)
data_module = my_load_dataset(config.dataset)
trainer = load_trainer(config)
trainer.fit(model=model, datamodule=data_module)
if checkpoint_path.exists():
    print('loading best checkpoint from', checkpoint_path)
    model.load_checkpoint(str(checkpoint_path))
else:
    print('best checkpoint was not found; testing the current model state')
trainer.test(model=model, datamodule=data_module)
print('test predictions saved to', TEST_RESULT_CSV)
print('model checkpoint path:', checkpoint_path)
try:
    from google.colab import files
    files.download(TEST_RESULT_CSV)
    if DOWNLOAD_MODEL_CHECKPOINT and checkpoint_path.exists():
        files.download(str(checkpoint_path))
except Exception:
    pass


In [ ]:
#@title 6. Predict classification/regression with a trained checkpoint
PREDICT_TASK_TYPE = 'classification' #@param ['classification', 'regression']
PREDICT_CSV = '' #@param {type:'string'}
UPLOAD_PREDICT_CSV = False #@param {type:'boolean'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
CHECKPOINT_PATH = '/content/SaprotHub/weights/prosst/ProSSTUserTask.pt' #@param {type:'string'}
PREDICT_NUM_LABELS = 2 #@param {type:'integer'}
PREDICT_BATCH_SIZE = 1 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
PREDICT_OUTPUT_CSV = '/content/prosst_predictions.csv' #@param {type:'string'}
DOWNLOAD_PREDICTION_RESULTS = True #@param {type:'boolean'}

from saprot.scripts.predict_prosst import predict_csv

PREDICT_CSV = maybe_upload_path(PREDICT_CSV, UPLOAD_PREDICT_CSV)
structure_base_dir = maybe_extract_asset_zip(STRUCTURE_ZIP, UPLOAD_STRUCTURE_ZIP)
if USE_LAST_STRUCTURE_TOKENS:
    PREDICT_CSV = attach_last_structure_tokens(PREDICT_CSV, '/content/prosst_predict_with_structure.csv')

pred_df = predict_csv(
    input_csv=PREDICT_CSV,
    output_csv=PREDICT_OUTPUT_CSV,
    task_type=PREDICT_TASK_TYPE,
    checkpoint_path=CHECKPOINT_PATH,
    model_path=MODEL_PATH,
    num_labels=PREDICT_NUM_LABELS,
    batch_size=PREDICT_BATCH_SIZE,
    cache_dir='/content/prosst_structure_cache',
    structure_base_dir=structure_base_dir,
)
display(pred_df.head())
print('saved to', PREDICT_OUTPUT_CSV)
if DOWNLOAD_PREDICTION_RESULTS:
    try:
        from google.colab import files
        files.download(PREDICT_OUTPUT_CSV)
    except Exception:
        pass


In [ ]:
#@title 7. Upload trained ColabProSST checkpoint to Hugging Face (optional)
HF_REPO_ID = '' #@param {type:'string'}
HF_PRIVATE_REPO = False #@param {type:'boolean'}
RUN_HF_LOGIN = True #@param {type:'boolean'}
CHECKPOINT_PATH = '/content/SaprotHub/weights/prosst/ProSSTUserTask.pt' #@param {type:'string'}
TASK_TYPE = 'classification' #@param ['classification', 'regression']
NUM_LABELS = 2 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
MODEL_CARD_TITLE = 'ColabProSST model' #@param {type:'string'}
MODEL_DESCRIPTION = 'A ProSST checkpoint trained with ColabProSST.' #@param {type:'string'}
DOWNLOAD_MODEL_PACKAGE = False #@param {type:'boolean'}

import json
import shutil
from pathlib import Path

checkpoint_path = Path(CHECKPOINT_PATH)
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint does not exist: {checkpoint_path}')
if not HF_REPO_ID.strip():
    raise ValueError('Set HF_REPO_ID, for example: username/Model-ProSST-Task')

if RUN_HF_LOGIN:
    from huggingface_hub import notebook_login
    notebook_login()

repo_id = HF_REPO_ID.strip()
package_root = Path('/content/SaprotHub/model_to_push/prosst')
package_dir = package_root / repo_id.replace('/', '__')
shutil.rmtree(package_dir, ignore_errors=True)
package_dir.mkdir(parents=True, exist_ok=True)

model_file = package_dir / 'model.pt'
shutil.copy2(checkpoint_path, model_file)
metadata = {
    'model_family': 'ProSST',
    'base_model': MODEL_PATH,
    'checkpoint_format': 'SaprotHub/ColabProSST torch checkpoint',
    'task_type': TASK_TYPE,
    'input_format': 'amino-acid input_ids + ProSST ss_input_ids',
    'structure_vocab_size': 2048,
    'colab_tool': 'ColabProSST',
}
if TASK_TYPE == 'classification':
    metadata['num_labels'] = int(NUM_LABELS)
(package_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
readme = f'''---
library_name: pytorch
base_model: {MODEL_PATH}
tags:
- protein-language-model
- ProSST
- ColabProSST
---

# {MODEL_CARD_TITLE}

{MODEL_DESCRIPTION}

This repository contains a SaprotHub/ColabProSST checkpoint (`model.pt`) and metadata for a ProSST downstream model.

## Input Format

ColabProSST uses amino-acid tokenizer `input_ids` together with ProSST structure token `ss_input_ids`. It does not use SaProt AA+3Di merged tokens.

## Task

- Task type: `{TASK_TYPE}`
- Base model: `{MODEL_PATH}`

Use `saprot/scripts/predict_prosst.py` from the ColabProSST branch to run prediction with this checkpoint.
'''
(package_dir / 'README.md').write_text(readme, encoding='utf-8')

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=repo_id, private=HF_PRIVATE_REPO, exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    folder_path=str(package_dir),
    commit_message='Upload ColabProSST checkpoint',
)
print('uploaded to', f'https://huggingface.co/{repo_id}')

if DOWNLOAD_MODEL_PACKAGE:
    archive_path = shutil.make_archive(str(package_dir), 'zip', package_dir)
    try:
        from google.colab import files
        files.download(archive_path)
    except Exception:
        print('model package zip:', archive_path)
